In [ ]:
! pip install datasets transformers seqeval

In [ ]:
!pip install evaluate

В этом блокноте мы дообучаем модель для задачи классификации токенов, а именно — распознавание именованных сущностей (NER). Мы используем датасет с медицинскими сущностями. Для ускорения работы используется компактная версия BERT для русского языка [rubert-tiny](https://huggingface.co/cointegrated/rubert-tiny).

Блокнот предназначен для работы с любой задачей токен-классификации, если модель имеет токен-классификационную голову и быстрый токенизатор. Возможно, потребуются небольшие корректировки при использовании другого датасета. Если возникает ошибка недостатка памяти, уменьшите размер батча.


*Ключевые шаги:*

  1. Подготовка данных обучения и сопоставление меток
  2. Загрузка предварительно обученной модели BERT и токенизатора
  3. Определение аргументов обучения и тренера
  4. Точная настройка модели на данных обучения
  5. Оценка на данных проверким




In [ ]:
model_checkpoint = "cointegrated/rubert-tiny"
batch_size = 16

## Загрузка датасета

Для обучения мы используем [Russian Drug Reaction Corpus](https://github.com/cimm-kzn/RuDReC): размеченный корпус отзывов на лекарства. Загрузка происходит с использованием библиотеки corus.

In [ ]:
from datasets import load_dataset

In [ ]:
!wget https://github.com/cimm-kzn/RuDReC/raw/master/data/rudrec_annotated.json
!pip install corus razdel

In [ ]:
from corus import load_rudrec
drugs = list(load_rudrec('rudrec_annotated.json'))
print(len(drugs))

Пример документа:

In [ ]:
drugs[0]

Посмотрим, какие сущности присутствуют: лекарства, форма лекарств, класс лекарств, показания к применению, побочные эффекты, а также болезни/симптомы.

In [ ]:
from collections import Counter, defaultdict
type2text = defaultdict(Counter)
ents = Counter()
for item in drugs:
    for e in item.entities:
        ents[e.entity_type] += 1
        type2text[e.entity_type][e.entity_text] += 1

for k, v in ents.most_common():
    print(k, v)
    print(type2text[k].most_common(3))

In [ ]:
drugs[0].text

Напишем функцию для преобразования разметки сущностей на уровень слов с использованием IOB-нотации.

In [ ]:
from razdel import tokenize

def extract_labels(item):
    raw_toks = list(tokenize(item.text))
    words = [tok.text for tok in raw_toks]
    word_labels = ['O'] * len(raw_toks)
    char2word = [None] * len(item.text)
    for i, word in enumerate(raw_toks):
        char2word[word.start:word.stop] = [i] * len(word.text)

    for e in item.entities:
        e_words = sorted({idx for idx in char2word[e.start:e.end] if idx is not None})
        word_labels[e_words[0]] = 'B-' + e.entity_type
        for idx in e_words[1:]:
            word_labels[idx] = 'I-' + e.entity_type

    return {'tokens': words, 'tags': word_labels}

In [ ]:
print(extract_labels(drugs[0]))

In [ ]:
from sklearn.model_selection import train_test_split
ner_data = [extract_labels(item) for item in drugs]
ner_train, ner_test = train_test_split(ner_data, test_size=0.2, random_state=1)

Пример данных:

In [ ]:
import pandas as pd
pd.options.display.max_colwidth = 300
pd.DataFrame(ner_train).sample(3)

Собираем все виды меток в список:

In [ ]:
label_list = sorted({label for item in ner_train for label in item['tags']})
if 'O' in label_list:
    label_list.remove('O')
    label_list = ['O'] + label_list
label_list

Создаём объект DatasetDict для обучения, валидации и тестирования:

In [ ]:
from datasets import Dataset, DatasetDict

ner_data = DatasetDict({
    'train': Dataset.from_pandas(pd.DataFrame(ner_train)),
    'test': Dataset.from_pandas(pd.DataFrame(ner_test))
})
ner_data

## Предобработка данных

Перед подачей текстов в модель их нужно токенизировать. Для этого используется токенизатор из библиотеки 🤗 Transformers, который преобразует слова в идентификаторы и генерирует дополнительные необходимые входы.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(...)

Можно напрямую вызвать токенизатор на предложении:

In [ ]:
tokenizer("Привет, это одно предложение!")

Если вход уже разделён на слова, передайте список слов в токенизатор с аргументом `is_split_into_words=True`:

In [ ]:
tokenizer(["Привет", ",", "это", "одно", "предложение", "разделённое", "на", "слова", "."], is_split_into_words=True)

Так как токенизатор может делить слова на субслова, ниже пример такого преобразования:

In [ ]:
example = ner_train[5]
print(example["tokens"])

In [ ]:
tokenized_input = tokenizer(example["tokens"], is_split_into_words=True)
tokens = tokenizer.convert_ids_to_tokens(tokenized_input["input_ids"])
print(tokens)

Токенизатор возвращает метод `word_ids`, который помогает соотнести субслова с исходными словами.

In [ ]:
print(tokenized_input.word_ids())

In [ ]:
word_ids = tokenized_input.word_ids()
aligned_labels = [-100 if i is None else example["tags"][i] for i in word_ids]
print(len(aligned_labels), len(tokenized_input["input_ids"]))

Определим функцию для токенизации и выравнивания меток:

In [ ]:
def tokenize_and_align_labels(examples, label_all_tokens=True):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples['tags']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx] if label_all_tokens else -100)
            previous_word_idx = word_idx

        label_ids = [label_list.index(idx) if isinstance(idx, str) else idx for idx in label_ids]

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

Применяем функцию к нашему датасету:

In [ ]:
tokenized_datasets = ner_data.map(tokenize_and_align_labels, batched=True)

## Дообучение модели

Скачиваем предобученную модель для токен-классификации и задаём количество меток:

In [ ]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

model = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=len(label_list))
model.config.id2label = dict(enumerate(label_list))
model.config.label2id = {v: k for k, v in model.config.id2label.items()}

Определяем параметры обучения:

In [ ]:
args = TrainingArguments(
    "ner",
    eval_strategy = ...,
    learning_rate=2e-5,
    per_device_train_batch_size= ...,
    per_device_eval_batch_size=...,
    num_train_epochs=10,
    weight_decay=0.01,
    save_strategy='no',
    report_to='none'
)

Создаём data collator для выравнивания размеров батчей:

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

Определяем метрику с помощью библиотеки seqeval:

In [ ]:
import evaluate
metric = evaluate.load("seqeval")

Функция для вычисления метрик:

In [ ]:
import numpy as np

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels, zero_division=0)
    return {
        "precision": ...,
        "recall": ...,
        "f1": ...,
        "accuracy": ...,
    }

Инициализируем Trainer:

In [ ]:
trainer = Trainer(
....

)

In [ ]:
trainer.evaluate()

На начальном этапе обучения заморозим все параметры модели, кроме последнего слоя:

In [ ]:
for param in model.bert.parameters():
    param.requires_grad = False

In [ ]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)
        print(param)

Обучаем модель:

In [ ]:
trainer.train()

После начального обучения разморозим все параметры и дообучим модель:

In [ ]:
# Разморозка всех параметров
for param in model.parameters():
    param.requires_grad = True

In [ ]:
args = TrainingArguments(
    "ner",
    eval_strategy = "epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=...,
    per_device_eval_batch_size=....,
    num_train_epochs=20,
    weight_decay=0.01,
    save_strategy='no',
    report_to='none'
)

In [ ]:
trainer = Trainer(
    model,
    args,
    train_dataset=...,
    eval_dataset=...,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Оценка модели после дообучения:

In [ ]:
trainer.evaluate()

Получение результатов предсказания и вычисление метрик:

In [ ]:
predictions, labels, _ = trainer.predict(tokenized_datasets["test"])
predictions = np.argmax(predictions, axis=2)

true_predictions = [
    [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]
true_labels = [
    [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]

results = metric.compute(predictions=true_predictions, references=true_labels)
results

Построение матрицы ошибок:

In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = pd.DataFrame(
    confusion_matrix(sum(true_labels, []), sum(true_predictions, []), labels=label_list),
    index=label_list,
    columns=label_list
)
cm

In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=cm)